In [ ]:
# ============================================================
# IBM SkillsBuild Data Analytics with AI Internship Project
# Author  : Rakesh Kumar Maity
# Project : Sales & Marketing Data Analytics
# Dataset : Sales& Marketing.csv
# ============================================================

In [ ]:
# ---- STEP 1: Import Required Libraries ----
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

In [ ]:
# ---- STEP 2: Load the Dataset ----
df = pd.read_csv('dataset/Sales& Marketing.csv', encoding='latin-1')
print('Dataset loaded successfully!')
print('Shape:', df.shape)

In [ ]:
# ---- STEP 3: Preview the Data ----
df.head(10)

In [ ]:
# ---- STEP 4: Dataset Info – Column Types and Non-Null Counts ----
df.info()

In [ ]:
# ---- STEP 5: Summary Statistics ----
df.describe(include='all')

In [ ]:
# ---- STEP 6: Check for Missing Values ----
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values found.')

In [ ]:
# ---- STEP 7: Check for Duplicate Rows ----
dup_count = df.duplicated().sum()
print(f'Duplicate rows: {dup_count}')

In [ ]:
# ---- STEP 8: Drop Duplicate Rows (if any) ----
df.drop_duplicates(inplace=True)
print(f'Shape after removing duplicates: {df.shape}')

In [ ]:
# ---- STEP 9: Normalize Column Names ----
# Strip whitespace and make lowercase for safe access
df.columns = df.columns.str.strip()
print('Columns:', df.columns.tolist())

In [ ]:
# ---- STEP 10: Parse Date Columns ----
# Identify date columns automatically
date_cols = [c for c in df.columns if 'date' in c.lower() or 'Date' in c]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], infer_datetime_format=True, errors='coerce')
print('Date columns parsed:', date_cols)

In [ ]:
# ---- STEP 11: Extract Year and Month from Order Date ----
order_date_col = [c for c in df.columns if 'order' in c.lower() and 'date' in c.lower()]
if order_date_col:
    odc = order_date_col[0]
    df['Order Year']  = df[odc].dt.year
    df['Order Month'] = df[odc].dt.month
    df['Month Name']  = df[odc].dt.strftime('%b')
    print(f"Year range: {df['Order Year'].min()} – {df['Order Year'].max()}")

In [ ]:
# ---- STEP 12: Ensure Numeric Columns are Correct Type ----
num_cols = ['Sales', 'Profit', 'Quantity', 'Discount']
for col in num_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        
# Fill remaining NaN in numeric cols with 0
df[num_cols] = df[num_cols].fillna(0)
print('Numeric columns verified.')

In [ ]:
# ---- STEP 13: Add Derived Column – Profit Margin (%) ----
df['Profit Margin (%)'] = np.where(
    df['Sales'] != 0,
    (df['Profit'] / df['Sales']) * 100,
    0
)
df['Profit Margin (%)'] = df['Profit Margin (%)'].round(2)
print('Profit Margin column created.')

In [ ]:
# ---- STEP 14: Key Business Metrics ----
total_sales   = df['Sales'].sum()
total_profit  = df['Profit'].sum()
total_orders  = df.shape[0]
avg_discount  = df['Discount'].mean() * 100
overall_margin = (total_profit / total_sales) * 100

print(f'Total Sales     : ${total_sales:,.2f}')
print(f'Total Profit    : ${total_profit:,.2f}')
print(f'Total Orders    : {total_orders:,}')
print(f'Avg Discount    : {avg_discount:.2f}%')
print(f'Overall Margin  : {overall_margin:.2f}%')

In [ ]:
# ---- STEP 15: Yearly Sales Trend ----
yearly_sales = df.groupby('Order Year')['Sales'].sum().reset_index()

plt.figure(figsize=(9, 5))
plt.plot(yearly_sales['Order Year'], yearly_sales['Sales'],
         marker='o', linewidth=2.5, color='steelblue')
for _, row in yearly_sales.iterrows():
    plt.text(row['Order Year'], row['Sales'] + yearly_sales['Sales'].max() * 0.01,
             f"${row['Sales']:,.0f}", ha='center', fontsize=9)
plt.title('Yearly Sales Trend')
plt.xlabel('Year')
plt.ylabel('Total Sales ($)')
plt.gca().yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.savefig('yearly_sales_trend.png', dpi=150)
plt.show()

In [ ]:
# ---- STEP 16: Monthly Sales Trend (all years combined) ----
month_order = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']
monthly_sales = df.groupby('Month Name')['Sales'].sum().reindex(month_order)

plt.figure(figsize=(11, 5))
bars = plt.bar(monthly_sales.index, monthly_sales.values,
               color=sns.color_palette('muted', 12))
for bar in bars:
    plt.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + monthly_sales.max() * 0.005,
             f'${bar.get_height():,.0f}', ha='center', va='bottom', fontsize=8)
plt.title('Monthly Sales Distribution')
plt.xlabel('Month')
plt.ylabel('Total Sales ($)')
plt.gca().yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.savefig('monthly_sales.png', dpi=150)
plt.show()

In [ ]:
# ---- STEP 17: Category-Wise Sales ----
cat_sales = df.groupby('Category')['Sales'].sum().sort_values(ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(x=cat_sales.index, y=cat_sales.values, palette='Set2')
for i, val in enumerate(cat_sales.values):
    plt.text(i, val + total_sales * 0.003, f'${val:,.0f}', ha='center', fontsize=10)
plt.title('Sales by Category')
plt.xlabel('Category')
plt.ylabel('Total Sales ($)')
plt.gca().yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.savefig('category_sales.png', dpi=150)
plt.show()

In [ ]:
# ---- STEP 18: Category-Wise Profit ----
cat_profit = df.groupby('Category')['Profit'].sum().sort_values(ascending=False)

plt.figure(figsize=(8, 5))
colors = ['#2ecc71' if v >= 0 else '#e74c3c' for v in cat_profit.values]
plt.bar(cat_profit.index, cat_profit.values, color=colors, edgecolor='white')
for i, val in enumerate(cat_profit.values):
    plt.text(i, val + (cat_profit.max() * 0.01 if val >= 0 else cat_profit.min() * 0.01),
             f'${val:,.0f}', ha='center', fontsize=10)
plt.title('Profit by Category')
plt.xlabel('Category')
plt.ylabel('Total Profit ($)')
plt.gca().yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.savefig('category_profit.png', dpi=150)
plt.show()

In [ ]:
# ---- STEP 19: Sub-Category Sales (Top 10) ----
subcat_sales = df.groupby('Sub-Category')['Sales'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(11, 5))
sns.barplot(x=subcat_sales.index, y=subcat_sales.values, palette='Blues_d')
plt.title('Top 10 Sub-Categories by Sales')
plt.xlabel('Sub-Category')
plt.ylabel('Total Sales ($)')
plt.xticks(rotation=30, ha='right')
plt.gca().yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.savefig('subcat_sales.png', dpi=150)
plt.show()

In [ ]:
# ---- STEP 20: Region-Wise Sales ----
region_sales = df.groupby('Region')['Sales'].sum().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Bar chart
sns.barplot(ax=axes[0], x=region_sales.index, y=region_sales.values, palette='Paired')
axes[0].set_title('Region-Wise Sales (Bar)')
axes[0].set_xlabel('Region')
axes[0].set_ylabel('Total Sales ($)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
for i, val in enumerate(region_sales.values):
    axes[0].text(i, val + region_sales.max()*0.01,
                 f'${val:,.0f}', ha='center', fontsize=9)

# Pie chart
axes[1].pie(region_sales.values, labels=region_sales.index,
            autopct='%1.1f%%', startangle=140,
            colors=sns.color_palette('Paired', len(region_sales)))
axes[1].set_title('Region-Wise Sales (Pie)')

plt.tight_layout()
plt.savefig('region_sales.png', dpi=150)
plt.show()

In [ ]:
# ---- STEP 21: Region-Wise Profit ----
region_profit = df.groupby('Region')['Profit'].sum().sort_values(ascending=False)

plt.figure(figsize=(9, 5))
colors = ['#27ae60' if v >= 0 else '#e74c3c' for v in region_profit.values]
plt.bar(region_profit.index, region_profit.values, color=colors, edgecolor='white')
for i, val in enumerate(region_profit.values):
    plt.text(i, val + region_profit.max()*0.01,
             f'${val:,.0f}', ha='center', fontsize=10)
plt.title('Profit by Region')
plt.xlabel('Region')
plt.ylabel('Total Profit ($)')
plt.gca().yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.savefig('region_profit.png', dpi=150)
plt.show()

In [ ]:
# ---- STEP 22: Top 10 Products by Sales ----
top_products_sales = (df.groupby('Product Name')['Sales']
                        .sum()
                        .sort_values(ascending=False)
                        .head(10))

plt.figure(figsize=(11, 6))
sns.barplot(x=top_products_sales.values, y=top_products_sales.index, palette='rocket_r')
plt.title('Top 10 Products by Sales')
plt.xlabel('Total Sales ($)')
plt.ylabel('Product Name')
plt.gca().xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.savefig('top10_products_sales.png', dpi=150)
plt.show()

In [ ]:
# ---- STEP 23: Top 10 Products by Profit ----
top_products_profit = (df.groupby('Product Name')['Profit']
                         .sum()
                         .sort_values(ascending=False)
                         .head(10))

plt.figure(figsize=(11, 6))
sns.barplot(x=top_products_profit.values, y=top_products_profit.index, palette='mako_r')
plt.title('Top 10 Products by Profit')
plt.xlabel('Total Profit ($)')
plt.ylabel('Product Name')
plt.gca().xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.savefig('top10_products_profit.png', dpi=150)
plt.show()

In [ ]:
# ---- STEP 24: Segment-Wise Sales & Profit ----
seg_data = df.groupby('Segment')[['Sales', 'Profit']].sum().reset_index()
seg_data = seg_data.sort_values('Sales', ascending=False)

x = np.arange(len(seg_data))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width/2, seg_data['Sales'],  width, label='Sales',  color='#3498db')
bars2 = ax.bar(x + width/2, seg_data['Profit'], width, label='Profit', color='#2ecc71')
ax.set_title('Segment-Wise Sales vs Profit')
ax.set_xlabel('Segment')
ax.set_ylabel('Amount ($)')
ax.set_xticks(x)
ax.set_xticklabels(seg_data['Segment'])
ax.legend()
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:,.0f}'))
plt.tight_layout()
plt.savefig('segment_sales_profit.png', dpi=150)
plt.show()

In [ ]:
# ---- STEP 25: Discount vs Profit Scatter Plot ----
plt.figure(figsize=(9, 5))
plt.scatter(df['Discount'], df['Profit'],
            alpha=0.35, c=df['Profit'],
            cmap='RdYlGn', edgecolors='none', s=30)
plt.colorbar(label='Profit ($)')
plt.title('Discount vs Profit')
plt.xlabel('Discount')
plt.ylabel('Profit ($)')
plt.axhline(0, color='red', linestyle='--', linewidth=0.8, label='Break-even')
plt.legend()
plt.tight_layout()
plt.savefig('discount_vs_profit.png', dpi=150)
plt.show()

In [ ]:
# ---- STEP 26: Discount vs Sales Scatter Plot ----
plt.figure(figsize=(9, 5))
plt.scatter(df['Discount'], df['Sales'],
            alpha=0.35, color='cornflowerblue', edgecolors='none', s=30)
plt.title('Discount vs Sales')
plt.xlabel('Discount')
plt.ylabel('Sales ($)')
plt.tight_layout()
plt.savefig('discount_vs_sales.png', dpi=150)
plt.show()

In [ ]:
# ---- STEP 27: Profit Margin by Category ----
cat_margin = df.groupby('Category')['Profit Margin (%)'].mean().sort_values(ascending=False)

plt.figure(figsize=(8, 5))
colors = ['#27ae60' if v >= 0 else '#e74c3c' for v in cat_margin.values]
plt.bar(cat_margin.index, cat_margin.values, color=colors)
for i, val in enumerate(cat_margin.values):
    plt.text(i, val + 0.3, f'{val:.1f}%', ha='center', fontsize=10)
plt.title('Average Profit Margin (%) by Category')
plt.xlabel('Category')
plt.ylabel('Avg Profit Margin (%)')
plt.tight_layout()
plt.savefig('profit_margin_category.png', dpi=150)
plt.show()

In [ ]:
# ---- STEP 28: Correlation Heatmap ----
corr_cols = ['Sales', 'Quantity', 'Discount', 'Profit', 'Profit Margin (%)']
corr_matrix = df[corr_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, square=True)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150)
plt.show()

In [ ]:
# ---- STEP 29: Ship Mode Distribution ----
ship_counts = df['Ship Mode'].value_counts()

plt.figure(figsize=(8, 5))
ship_counts.plot(kind='bar', color=sns.color_palette('pastel', len(ship_counts)), edgecolor='grey')
plt.title('Orders by Ship Mode')
plt.xlabel('Ship Mode')
plt.ylabel('Number of Orders')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig('ship_mode.png', dpi=150)
plt.show()

In [ ]:
# ---- STEP 30: Top 10 States by Sales ----
state_sales = (df.groupby('State')['Sales']
                 .sum()
                 .sort_values(ascending=False)
                 .head(10))

plt.figure(figsize=(11, 5))
sns.barplot(x=state_sales.index, y=state_sales.values, palette='viridis')
plt.title('Top 10 States by Sales')
plt.xlabel('State')
plt.ylabel('Total Sales ($)')
plt.xticks(rotation=30, ha='right')
plt.gca().yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.savefig('top10_states.png', dpi=150)
plt.show()

In [ ]:
# ---- STEP 31: Bottom 5 Loss-Making Sub-Categories ----
subcat_profit = (df.groupby('Sub-Category')['Profit']
                   .sum()
                   .sort_values()
                   .head(5))

plt.figure(figsize=(9, 5))
plt.barh(subcat_profit.index, subcat_profit.values, color='#e74c3c', edgecolor='white')
for i, val in enumerate(subcat_profit.values):
    plt.text(val - subcat_profit.abs().max()*0.02, i, f'${val:,.0f}',
             va='center', ha='right', color='white', fontweight='bold', fontsize=9)
plt.title('Bottom 5 Sub-Categories by Profit (Loss-Making)')
plt.xlabel('Total Profit ($)')
plt.tight_layout()
plt.savefig('loss_subcat.png', dpi=150)
plt.show()

In [ ]:
# ---- STEP 32: Sales vs Profit Scatter by Category ----
category_colors = {'Furniture': '#e67e22',
                   'Office Supplies': '#3498db',
                   'Technology': '#2ecc71'}

plt.figure(figsize=(10, 6))
for cat, grp in df.groupby('Category'):
    color = category_colors.get(cat, 'grey')
    plt.scatter(grp['Sales'], grp['Profit'],
                label=cat, alpha=0.4, s=25,
                color=color, edgecolors='none')
plt.axhline(0, color='red', linestyle='--', linewidth=0.8)
plt.title('Sales vs Profit by Category')
plt.xlabel('Sales ($)')
plt.ylabel('Profit ($)')
plt.legend(title='Category')
plt.tight_layout()
plt.savefig('sales_vs_profit_category.png', dpi=150)
plt.show()

In [ ]:
# ---- STEP 33: Quantity Distribution by Category ----
plt.figure(figsize=(9, 5))
for cat in df['Category'].unique():
    sns.kdeplot(df[df['Category'] == cat]['Quantity'],
                label=cat, fill=True, alpha=0.3)
plt.title('Quantity Distribution by Category')
plt.xlabel('Quantity')
plt.ylabel('Density')
plt.legend(title='Category')
plt.tight_layout()
plt.savefig('quantity_dist.png', dpi=150)
plt.show()

In [ ]:
# ---- STEP 34: Yearly Sales & Profit Comparison ----
yearly_sp = df.groupby('Order Year')[['Sales', 'Profit']].sum().reset_index()

x = np.arange(len(yearly_sp))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width/2, yearly_sp['Sales'],  width, label='Sales',  color='#2980b9')
ax.bar(x + width/2, yearly_sp['Profit'], width, label='Profit', color='#27ae60')
ax.set_title('Yearly Sales vs Profit Comparison')
ax.set_xlabel('Year')
ax.set_ylabel('Amount ($)')
ax.set_xticks(x)
ax.set_xticklabels(yearly_sp['Order Year'])
ax.legend()
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:,.0f}'))
plt.tight_layout()
plt.savefig('yearly_sales_profit.png', dpi=150)
plt.show()

In [ ]:
# ---- STEP 35: Pivot Table – Category x Region Sales ----
pivot = df.pivot_table(values='Sales', index='Category',
                       columns='Region', aggfunc='sum')

plt.figure(figsize=(9, 4))
sns.heatmap(pivot, annot=True, fmt=',.0f', cmap='YlOrRd',
            linewidths=0.5)
plt.title('Sales Heatmap: Category vs Region')
plt.tight_layout()
plt.savefig('pivot_heatmap.png', dpi=150)
plt.show()

In [ ]:
# ---- STEP 36: Sales & Profit Summary Table ----
summary = df.groupby('Category').agg(
    Total_Sales   = ('Sales',              'sum'),
    Total_Profit  = ('Profit',             'sum'),
    Total_Orders  = ('Sales',              'count'),
    Avg_Discount  = ('Discount',           'mean'),
    Avg_Margin_Pct= ('Profit Margin (%)',  'mean')
).round(2).reset_index()

summary['Total_Sales']  = summary['Total_Sales'].map('${:,.2f}'.format)
summary['Total_Profit'] = summary['Total_Profit'].map('${:,.2f}'.format)
print(summary.to_string(index=False))

In [ ]:
# ---- STEP 37: Final Project Summary ----
print('=' * 55)
print('     SALES & MARKETING – PROJECT SUMMARY')
print('=' * 55)
print(f'  Total Records Analyzed : {df.shape[0]:,}')
print(f'  Total Sales Revenue    : ${total_sales:,.2f}')
print(f'  Total Profit Earned    : ${total_profit:,.2f}')
print(f'  Overall Profit Margin  : {overall_margin:.2f}%')
print(f'  Average Discount Given : {avg_discount:.2f}%')
print(f'  Best Category (Sales)  : {cat_sales.idxmax()}')
print(f'  Best Region (Sales)    : {region_sales.idxmax()}')
print(f'  Best Region (Profit)   : {region_profit.idxmax()}')
print('=' * 55)
print('  Analysis complete. All charts saved as .png files.')
print('=' * 55)